In [6]:
import numpy as np

In [ ]:
def make_regression_1d(n=120, noise=0.3, outliers=10, seed=0):
    """
    y = 1 + 2x + Gaussian noise, plus some outliers (to stress-test MSE vs MAE/Huber).
    Returns: X (n,2) with bias, y (n,)
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2.5, 2.5, size=n)  # Gives us a list of randon x values
    y = 1.0 + 2.0*x + rng.normal(0.0, noise, size=n) # gives us the list of y values with noise added

    # add outliers
    idx = rng.choice(n, size=outliers, replace=False) # select x values to add noise
    y[idx] += rng.normal(0.0, 6.0, size=outliers)  # big spikes
    # np.c_ means column stack
    X = np.c_[np.ones(n), x]  # bias + feature
    return X, y

In [ ]:
# Just  an experiment
rng = np.random.default_rng(0)
x = rng.uniform(-2.5, 2.5, 5)
print(x)
y = 2*x 
print(y)
print(y/x)
y += rng.normal(0.0, 0.3, 5)
print(y)
print(y/x)

[ 0.68480844 -1.15106643 -2.29513238 -2.41736182  1.5663512 ]
[ 1.36961687 -2.30213286 -4.59026476 -4.83472364  3.13270239]
[2. 2. 2. 2. 2.]
[ 1.47809539 -1.91093285 -4.30614047 -5.04584422  2.75307595]
[2.1584071  1.66014124 1.87620571 2.08733511 1.75763645]


In [ ]:
import numpy as np

def make_binary_classification_2d_verbose(n=300, seed=0):
    """
    Create a simple 2D binary classification dataset.

    Idea:
    - We create two clusters ("Gaussian blobs") in 2D:
        Class 0: points around (-1.2, -1.0)
        Class 1: points around (+1.2, +1.0)
    - Each cluster is sampled from a normal distribution.
    - Then we stack them into one dataset, create labels,
      shuffle everything, and add a bias column.

    Returns:
    - X: shape (n, 3)  -> [1, x1, x2] (bias + 2 features)
    - y: shape (n,)     -> labels in {0, 1}
    """

    # Random number generator (reproducible if seed is fixed)
    rng = np.random.default_rng(seed)

    # Split n samples into two classes
    n0 = n // 2       # class 0 gets half (integer division)
    n1 = n - n0       # class 1 gets the rest (handles odd n)

    # --- Generate 2D points for class 0 ---
    # size=(n0,2) means: n0 points, each with 2 coordinates (x1,x2)
    # loc=(-1.2, -1.0) means: mean vector (center of the blob)
    # scale=0.8 means: standard deviation (spread) in each dimension
    X0 = rng.normal(loc=(-1.2, -1.0), scale=0.8, size=(n0, 2))

    # --- Generate 2D points for class 1 ---
    X1 = rng.normal(loc=(+1.2, +1.0), scale=0.8, size=(n1, 2))

    # Stack the two groups of points on top of each other
    # X will be shape (n,2)
    X = np.vstack([X0, X1])

    # Create labels:
    # - first n0 points belong to class 0 -> label 0
    # - next n1 points belong to class 1 -> label 1
    y0 = np.zeros(n0, dtype=int)
    y1 = np.ones(n1, dtype=int)
    y = np.concatenate([y0, y1])   # shape (n,)

    # At this point, the dataset is "ordered":
    # first all class 0, then all class 1.
    # Shuffling prevents your training loop from seeing classes in blocks.

    # perm is a random permutation of indices [0, 1, ..., n-1]
    perm = rng.permutation(n)

    # Reorder X and y with the same permutation so labels stay aligned
    X = X[perm]
    y = y[perm]

    # Add bias column:
    # We turn X from shape (n,2) into shape (n,3) by adding a first column of ones.
    # This allows a linear classifier to learn an intercept term.
    X = np.c_[np.ones(n), X]  # [1, x1, x2]

    return X, y


In [29]:
# Another experiment
# print(make_binary_classification_2d())
# X0 = rng.normal(loc=(-1.2, -1.0), scale=0.8, size=(n0, 2))

import numpy as np
rng = np.random.default_rng(0)

n0 = 5
X0 = rng.normal(loc=(-1.2, -1.0), scale=0.8, size=(n0, 2))

print("X0 shape:", X0.shape)
print(X0)


n1 = 5
X1 = rng.normal(loc=(+1.2, +1.0), scale=0.8, size=(n1, 2))

print("X1 shape:", X1.shape)
print(X1)

X = np.vstack([X0, X1])
print("X shape:", X.shape)
print(X)

y0 = np.zeros(n0, dtype=int)
y1 = np.ones(n1, dtype=int)
y = np.concatenate([y0, y1])

print("y shape:", y.shape)
print(y)

perm = rng.permutation(n0 + n1)
print("perm:", perm)

X_shuf = X[perm]
y_shuf = y[perm]

print("shuffled labels:", y_shuf)

n = X_shuf.shape[0]
X_bias = np.c_[np.ones(n), X_shuf]

print("X_shuf shape:", X_shuf.shape)
print("X_bias shape:", X_bias.shape)
print("first 3 rows:\n", X_bias[:3])


X0 shape: (5, 2)
[[-1.09941582 -1.10568389]
 [-0.68766188 -0.91607991]
 [-1.6285355  -0.71072396]
 [-0.15679996 -0.24233523]
 [-1.76298819 -2.01233718]]
X1 shape: (5, 2)
[[ 0.70138043  1.03306078]
 [-0.66002462  0.82496667]
 [ 0.20327124  0.41418612]
 [ 0.76459281  0.74695987]
 [ 1.52930443  1.8340107 ]]
X shape: (10, 2)
[[-1.09941582 -1.10568389]
 [-0.68766188 -0.91607991]
 [-1.6285355  -0.71072396]
 [-0.15679996 -0.24233523]
 [-1.76298819 -2.01233718]
 [ 0.70138043  1.03306078]
 [-0.66002462  0.82496667]
 [ 0.20327124  0.41418612]
 [ 0.76459281  0.74695987]
 [ 1.52930443  1.8340107 ]]
y shape: (10,)
[0 0 0 0 0 1 1 1 1 1]
perm: [4 5 7 6 2 3 1 9 0 8]
shuffled labels: [0 1 1 1 0 0 0 1 0 1]
X_shuf shape: (10, 2)
X_bias shape: (10, 3)
first 3 rows:
 [[ 1.         -1.76298819 -2.01233718]
 [ 1.          0.70138043  1.03306078]
 [ 1.          0.20327124  0.41418612]]


This little class is implementing the Mean Squared Error (MSE) loss for regression, plus the gradient you need for gradient descent.
1) What it represents mathematically

For one data point:

$$
e=\frac{1}{2}\left(y-y_T\right)^2
$$


For a dataset of $p$ points (what the code uses):

$$
E=\frac{1}{p} \sum_{\alpha=1}^p \frac{1}{2}\left(y^{(\alpha)}-y_T^{(\alpha)}\right)^2
$$


Where:
- $y^{(\alpha)}=$ prediction (y_pred)
- $y_T^{(\alpha)}=$ target ( y_true )
- $r^{(\alpha)}=y^{(\alpha)}-y_T^{(\alpha)}=$ residual

In [31]:
class MSE:
    def __call__(self, y_pred, y_true):
        r = y_pred - y_true
        loss = 0.5 * np.mean(r**2)
        dL_dy = r / len(y_true)
        return loss, dL_dy

This one is the Mean Absolute Error (MAE) loss, plus the (sub)gradient you need for gradient descent.
1) Math meaning

Per sample:

$$
e=\left|y-y_T\right|
$$


Dataset average:

$$
E=\frac{1}{p} \sum_{\alpha=1}^p\left|y^{(\alpha)}-y_T^{(\alpha)}\right|
$$


Define residuals $r^{(\alpha)}=y^{(\alpha)}-y_T^{(\alpha)}$. Then:

$$
E=\frac{1}{p} \sum_\alpha\left|r^{(\alpha)}\right|
$$

In [34]:
class MAE:
    def __call__(self, y_pred, y_true):
        r = y_pred - y_true
        loss = np.mean(np.abs(r))
        # subgradient: sign(r), define 0 at r=0
        d = np.sign(r)
        dL_dy = d / len(y_true)
        return loss, dL_dy

Huber is the "peace treaty" between MSE and MAE: it behaves like squared error for small mistakes (smooth, fast learning), and like absolute error for big mistakes (robust to outliers).
1) The math definition

Let residual $r=y-y_T$ and $\delta>0$ (your delta).
The Huber loss per sample is:

$$
e_\delta(r)= \begin{cases}\frac{1}{2} r^2 & \text { if }|r| \leq \delta \\ \delta\left(|r|-\frac{1}{2} \delta\right) & \text { if }|r|>\delta\end{cases}
$$

- Near 0: quadratic (like MSE)
- Far away: linear (like MAE)
- Continuous and differentiable at $|r|=\delta$

Dataset cost:

$$
E=\frac{1}{p} \sum_\alpha e_\delta\left(r^{(\alpha)}\right)
$$


Derivative w.r.t. $r$ :

$$
\frac{d e_\delta}{d r}= \begin{cases}r & |r| \leq \delta \\ \delta \operatorname{sign}(r) & |r|>\delta\end{cases}
$$


Since $r=y-y_T$, we also have $\partial E / \partial y=(1 / p) d e_\delta / d r$.

In [ ]:
class Huber:
    def __init__(self, delta=1.0):
        self.delta = float(delta)

    def __call__(self, y_pred, y_true):
        r = y_pred - y_true
        a = np.abs(r)
        quad = a <= self.delta

        loss = np.mean(np.where(quad, 0.5*r**2, self.delta*(a - 0.5*self.delta)))
        d = np.where(quad, r, self.delta*np.sign(r))
        dL_dy = d / len(y_true)
        return loss, dL_dy

This one is another "MSE-MAE compromise", like Huber, but fully smooth (no kink). It's based on the function $\log (\cosh (r))$.
1) What it implements (math)

Residual:

$$
r=y-y_T
$$


Per-sample loss:

$$
e(r)=\log (\cosh (r))
$$


Dataset loss:

$$
E=\frac{1}{p} \sum_{\alpha=1}^p \log \left(\cosh \left(r^{(\alpha)}\right)\right)
$$


Derivative w.r.t. $r$ :

$$
\frac{d}{d r} \log (\cosh (r))=\tanh (r)
$$


Because:
- $\frac{d}{d r} \cosh (r)=\sinh (r)$
- chain rule: $\frac{d}{d r} \log (\cosh (r))=\frac{1}{\cosh (r)} \sinh (r)=\tanh (r)$

Since $r=y-y_T$, we also have:

$$
\frac{\partial E}{\partial y^{(\alpha)}}=\frac{1}{p} \tanh \left(r^{(\alpha)}\right)
$$


That matches the code.

In [33]:
class LogCosh:
    def __call__(self, y_pred, y_true):
        r = y_pred - y_true
        # log(cosh(r)) is smooth-ish and behaves like L2 near 0, like L1 for large |r|
        loss = np.mean(np.log(np.cosh(r)))
        d = np.tanh(r)
        dL_dy = d / len(y_true)
        return loss, dL_dy

In [32]:
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

# Siehe Tut1

In [ ]:
class BCEWithLogits:
    """
    y in {0,1}, h = logits.
    loss = mean( softplus(h) - y*h )
    grad: dL/dh = sigmoid(h) - y
    """
    def __call__(self, h, y01):
        y01 = y01.astype(float)
        # stable softplus: log(1+exp(h))
        softplus = np.logaddexp(0.0, h)
        loss = np.mean(softplus - y01*h)
        dL_dh = (sigmoid(h) - y01) / len(y01)
        return loss, dL_dh

In [ ]:



class Hinge:
    """
    y in {-1,+1}, h = logits.
    loss = mean(max(0, 1 - y*h))
    grad: dL/dh = -y if margin violated else 0
    """
    def __call__(self, h, ypm1):
        ypm1 = ypm1.astype(float)
        margin = 1.0 - ypm1*h
        active = margin > 0
        loss = np.mean(np.maximum(0.0, margin))
        dL_dh = np.where(active, -ypm1, 0.0) / len(ypm1)
        return loss, dL_dh


In [4]:
def train_linear_regression(X, y, loss_fn, lr=0.1, epochs=2000, seed=0):
    rng = np.random.default_rng(seed)
    w = rng.normal(0.0, 0.01, size=X.shape[1])

    history = []
    for t in range(epochs):
        y_pred = X @ w
        loss, dL_dy = loss_fn(y_pred, y)
        # y_pred = Xw => dy/dw = X, so grad_w = X^T * dL/dy
        grad_w = X.T @ dL_dy
        w -= lr * grad_w
        history.append(loss)

    return w, np.array(history)


In [5]:
Xr, yr = make_regression_1d(n=140, noise=0.25, outliers=12, seed=1)

losses = {
    "MSE": MSE(),
    "MAE": MAE(),
    "Huber(d=1.0)": Huber(delta=1.0),
    "LogCosh": LogCosh(),
}

results = {}
for name, lf in losses.items():
    w, hist = train_linear_regression(Xr, yr, lf, lr=0.2, epochs=3000, seed=0)
    results[name] = (w, hist[-1])

for name, (w, last_loss) in results.items():
    print(f"{name:12s} | final loss {last_loss:.4f} | w = {w}")


MSE          | final loss 0.9083 | w = [1.06511506 2.01867091]
MAE          | final loss 0.4785 | w = [0.96411445 1.99140457]
Huber(d=1.0) | final loss 0.2949 | w = [0.9666654  2.00773907]
LogCosh      | final loss 0.2807 | w = [0.96585666 2.00519516]
